# Train: Sacral / Disc-Morphology head

Adds a 4th head to `LumbarDiagnosticModel` -- disc morphology
(Normal / Degenerated / Bulging / Herniated / Thinning / Disc
Degeneration with Osteophyte formation) for the lowest 3 disc levels
(L3/L4, L4/L5, L5/S1). L5/S1 is the "sacral" level per the project
brief, since the sacrum is one fused bone with no discs of its own
beyond that junction.

This is a **separate taxonomy** from the existing RSNA severity-grade
head (Normal/Mild, Moderate, Severe) -- both heads coexist on the same
model, trained jointly.

**Data:** Sudirman et al. "Lumbar Spine MRI Dataset" + companion
"Radiologists Notes" (Mendeley Data, **CC BY 4.0** -- free, just cite
the authors):
- Images: https://data.mendeley.com/datasets/k57fr854j2/2
- Notes:  https://data.mendeley.com/datasets/s6bgczr8s2/2

**No Google Drive is used anywhere in this notebook.** Files move
directly between your local machine and this Colab VM's own
(ephemeral, Drive-quota-free) disk.

**If you're running this through the VS Code Colab extension**: its
bridge to the Colab web UI is incomplete -- `google.colab.files.upload()`
and `files.download()` are both currently broken there (known upstream
issues, not something a retry fixes). This notebook uses the documented
workarounds instead: the plain `ipywidgets` File Upload widget for
uploads, and a base64 HTML download link for pulling the trained
checkpoint back. Both are plain Jupyter/browser mechanics, not
Colab-specific JS, so they work the same in VS Code as in the browser.

**Before running the training cell**, run the inspection cell below and
*read its output*. The exact structure of the radiologist-notes file
wasn't confirmed ahead of time -- if it doesn't match what
`SudirmanDiscDataset` expects (a `radiologist_notes.csv` with a
`study_id` column plus one free-text column per level), fix it up in
the "Fix layout if needed" cell before training, not after.

## 1. Runtime check

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'

## 2. Get the project code onto this Colab session

Both the Colab-native `files.upload()` widget and generic `ipywidgets`
render as inert/greyed-out through the VS Code Colab bridge (confirmed
broken, not a fluke -- known upstream limitation). `git clone` is a
plain HTTPS request, no widget/comm channel involved, so it sidesteps
the problem entirely.

The project code (not the datasets, not the full checkpoint) lives at
https://github.com/sameerkulk65/lumbar-mri-dx (public, so no token
needed).

In [ ]:
PROJECT_DIR = '/content/lumbar_mri_dx'
!git clone -q https://github.com/sameerkulk65/lumbar-mri-dx.git {PROJECT_DIR}
%cd {PROJECT_DIR}
!ls

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Download the Sudirman datasets

Same problem as section 2 (no working upload widget), same fix: get
the files onto GitHub first, then plain `wget` them in Colab -- no
widget, no token (public repo).

**On your local machine:**
1. Download both zips from Mendeley (still manual -- no stable
   scriptable URL):
   - Images: https://data.mendeley.com/datasets/k57fr854j2/2
   - Radiologists Notes: https://data.mendeley.com/datasets/s6bgczr8s2/2
2. Go to https://github.com/sameerkulk65/lumbar-mri-dx/releases/new
3. Set a tag (e.g. `sudirman-data`), drag both zip files into the
   "Attach binaries" area, click **Publish release**.
4. Right-click each attached file on the release page -> **Copy link
   address** -> paste them into `IMAGES_URL` / `NOTES_URL` below.

In [ ]:
IMAGES_URL = 'https://github.com/sameerkulk65/lumbar-mri-dx/releases/download/sudirman-data/sudirman_images.zip'  # <-- set this
NOTES_URL  = 'https://github.com/sameerkulk65/lumbar-mri-dx/releases/download/sudirman-data/sudirman_notes.zip'   # <-- set this

import os, zipfile

SUDIRMAN_DIR = f'{PROJECT_DIR}/data/sudirman'
os.makedirs(f'{SUDIRMAN_DIR}/images', exist_ok=True)

!wget -q "{IMAGES_URL}" -O /content/sudirman_images.zip
!wget -q "{NOTES_URL}" -O /content/sudirman_notes.zip

with zipfile.ZipFile('/content/sudirman_images.zip') as zf:
    zf.extractall(f'{SUDIRMAN_DIR}/images')
with zipfile.ZipFile('/content/sudirman_notes.zip') as zf:
    zf.extractall(SUDIRMAN_DIR)
print('Extracted to', SUDIRMAN_DIR)

## 5. INSPECT the real file layout -- read this output before continuing

`SudirmanDiscDataset` (in `src/spine_datasets.py`) currently expects:
- `data/sudirman/images/<study_id>/*.dcm` (or `.jpg`)
- `data/sudirman/radiologist_notes.csv` with a `study_id` column and
  one free-text note column per level (`l3_l4`, `l4_l5`, `l5_s1`)

That's a best-effort guess. Run this cell and actually look at the
output before trusting the parser.

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(SUDIRMAN_DIR)
print('--- Top-level contents ---')
for p in sorted(root.rglob('*'))[:40]:
    print(' ', p.relative_to(root))

print('\n--- Candidate annotation files ---')
for ext in ('*.csv', '*.txt', '*.xlsx', '*.json'):
    for f in root.rglob(ext):
        print(' ', f.relative_to(root))

notes_csv = root / 'radiologist_notes.csv'
if notes_csv.exists():
    df = pd.read_csv(notes_csv)
    print('\nradiologist_notes.csv columns:', list(df.columns))
    print(df.head(10))
else:
    print('\nNo radiologist_notes.csv at the expected path -- find the real')
    print('annotation file from the listing above and inspect it manually, e.g.:')
    print("  pd.read_csv(root / '<real_filename>').head(10)")

## 5b. Fix layout if needed

If cell 5's output doesn't match what `SudirmanDiscDataset` expects,
fix it *here* (rename/reshape files, or edit `MORPH_LEVELS` /
`parse_morph_note()` / `SudirmanDiscDataset` in
`src/spine_datasets.py` directly) before moving on. Common fixes:
- Real file uses a different column layout (e.g. one row per
  study+level instead of one row per study with 3 level columns) --
  pivot it into the wide format expected, and re-save as
  `radiologist_notes.csv`.
- Real note phrasing doesn't match the keyword lists in
  `MORPH_KEYWORDS` -- extend them based on what cell 5 printed.

In [ ]:
# (edit as needed once you've looked at cell 5's output)


## 6. Sanity check: class distribution

Confirms the parser produces a sane, non-degenerate label distribution
before spending GPU hours on it.

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)
import numpy as np
from src.spine_datasets import SudirmanDiscDataset, MORPH_LEVELS, MORPH_CLASSES

ds = SudirmanDiscDataset(SUDIRMAN_DIR, split='train')
counts = np.zeros((len(MORPH_LEVELS), len(MORPH_CLASSES)), dtype=int)
for i in range(len(ds)):
    _, target = ds[i]
    for lvl_idx, cls_idx in enumerate(target['morph_labels'].tolist()):
        counts[lvl_idx, cls_idx] += 1

names = list(MORPH_CLASSES.keys()) if isinstance(MORPH_CLASSES, dict) else MORPH_CLASSES
for lvl_idx, level in enumerate(MORPH_LEVELS):
    print(level, dict(zip(names, counts[lvl_idx].tolist())))

print('\nIf one class dominates >90% at every level, the keyword parser')
print('is probably not matching real note phrasing -- go back to cell 5b.')

## 7. Quick CPU-equivalent smoke test (fast on GPU here)

Confirms the model/loss/train wiring is correct on synthetic data
before touching real data or spending a long training run on a bug.

In [ ]:
!python src/train.py --mode smoke

## 7b. Safety net: per-epoch checkpoint download

If this Colab session disconnects mid-training (free-tier GPU quota
running out is the usual cause), everything under `/content/` --
including checkpoints -- is gone with it. Lightning already writes
`last.ckpt` after every epoch, but that alone doesn't help if the VM
dies before you reach the download cell at the end.

This callback re-saves a slim, download-ready checkpoint after every
epoch and displays a fresh clickable download link right in this
cell's output -- so as long as you grab the latest link before a
disconnect, you only lose progress back to the last completed epoch,
not the whole run. If epochs are fast and the repeated encoding feels
slow, raise `every_n_epochs` below.

In [ ]:
import base64, os, torch
from IPython.display import HTML, display
import lightning as L

class DownloadCheckpointCallback(L.Callback):
    def __init__(self, every_n_epochs=1):
        self.every_n_epochs = every_n_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch
        if (epoch + 1) % self.every_n_epochs != 0:
            return
        state = {'state_dict': pl_module.state_dict(), 'epoch': epoch}
        path = f'/content/checkpoint_epoch{epoch}.pt'
        torch.save(state, path)

        with open(path, 'rb') as f:
            data = f.read()
        b64 = base64.b64encode(data).decode()
        size_mb = len(data) / 1e6
        html = (
            '<div style="margin:6px 0;">Epoch {} checkpoint ready -- '
            '<a download="trained_state_dict_epoch{}.pt" '
            'href="data:application/octet-stream;base64,{}" '
            'style="padding:6px 14px;background:#34a853;color:#fff;'
            'border-radius:6px;text-decoration:none;">'
            'download ({:.1f} MB)</a></div>'
        ).format(epoch, epoch, b64, size_mb)
        display(HTML(html))
        os.remove(path)

## 8. Train

Initializes encoder/detection/segmentation/classification from your
existing `last.ckpt` (epoch 11) and trains the new morph_head (plus
continued fine-tuning of everything else) on GPU. Uses `init_from=`
(partial `strict=False` load), not `resume_ckpt=`, since the
architecture grew a head that isn't in the old checkpoint.

Watch this cell's output as it runs -- a fresh download link appears
after every epoch (see 7b above).

In [ ]:
import yaml
from src.train import train

with open('configs/config.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

best_ckpt = train(
    cfg,
    init_from=f'{PROJECT_DIR}/outputs/checkpoints/last_state_dict.pt',
    extra_callbacks=[DownloadCheckpointCallback(every_n_epochs=1)],
)
print('Best checkpoint:', best_ckpt)

## 9. Download the trained checkpoint back to your local machine

Run this once training finishes normally (if it got cut off instead,
just use the latest per-epoch link from section 8's output -- this
cell needs `best_ckpt`, which only exists after `train()` returns).

`google.colab.files.download()` doesn't work through the VS Code
bridge, so this builds a clickable browser-download link instead (plain
HTML `data:` URI -- no Colab-specific JS involved).

This also slims the checkpoint down first: `best_ckpt` is a full
PyTorch Lightning checkpoint (optimizer/scheduler state included, ~3x
the model's actual size). `export_trained_model.py` only needs the
`state_dict` + `epoch`, so we strip it down to just that before
downloading -- faster, and well under half the size.

In [ ]:
import torch

src_ckpt = best_ckpt or f'{PROJECT_DIR}/outputs/checkpoints/last.ckpt'
full = torch.load(src_ckpt, map_location='cpu')
slim = {'state_dict': full['state_dict'], 'epoch': full.get('epoch')}
SLIM_PATH = '/content/trained_state_dict.pt'
torch.save(slim, SLIM_PATH)

import os
print('Source checkpoint:', src_ckpt)
print('Slimmed artifact:', SLIM_PATH, '--',
      round(os.path.getsize(SLIM_PATH) / 1e6, 1), 'MB',
      '(was', round(os.path.getsize(src_ckpt) / 1e6, 1), 'MB)')

In [ ]:
import base64
from IPython.display import HTML, display

with open(SLIM_PATH, 'rb') as f:
    data = f.read()
b64 = base64.b64encode(data).decode()
size_mb = len(data) / 1e6
print(f'Encoding {size_mb:.1f} MB for download -- this cell may take a little while to render.')

html = (
    '<a download="trained_state_dict.pt" '
    'href="data:application/octet-stream;base64,{}" '
    'style="font-size:16px;padding:10px 18px;background:#4285f4;color:#fff;'
    'border-radius:6px;text-decoration:none;display:inline-block;">'
    'Click to download trained_state_dict.pt ({:.1f} MB)</a>'
).format(b64, size_mb)
display(HTML(html))

print()
print('After downloading, on your local machine:')
print('  1. Move it to outputs\\checkpoints\\trained_state_dict.pt (or anywhere)')
print('  2. python export_trained_model.py outputs\\checkpoints\\trained_state_dict.pt')
print('  3. Restart Streamlit -- the Sacral / Disc Morphology section')
print('     will now show real predictions instead of untrained ones.')